In [1]:
# ============================================================
# 01_preprocessing.py
# IBD Proteomic Subtyping Pipeline
# Supervisor-approved pipeline
# ============================================================

import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.decomposition import PCA
from scipy import stats
from scipy.stats import norm, ks_2samp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import os

# ── Config ────────────────────────────────────────
PROJECT_DIR  = '/rds/homes/j/jxt554'
DATA_DIR     = f'{PROJECT_DIR}/data'
FIGURES_DIR  = f'{PROJECT_DIR}/figures'
TABLES_DIR   = f'{PROJECT_DIR}/tables'
SEED         = 42
KNN_K        = 10

RAW_PROTEINS = ('/rds/projects/a/acharjea-multi-omics'
                '/proteomics/ukbb_ibd_proteins.tsv.gz')
RAW_METADATA = ('/rds/projects/a/acharjea-multi-omics'
                '/proteomics/ukbb_ibd_metadata.tsv')

for d in [DATA_DIR, FIGURES_DIR, TABLES_DIR]:
    os.makedirs(d, exist_ok=True)

print('IBD PROTEOMIC PREPROCESSING PIPELINE')
print('='*55)
print('Steps:')
print('  1. Load data')
print('  2. Remove IBD-unclassified')
print('  3. Derive batch from missingness')
print('  4. Protein QC (>30% missing → remove)')
print('  5. Sample QC  (>30% missing → remove)')
print('  6. Imputation comparison')
print('     KNN vs Median vs MICE')
print('     Select best via KS test')
print('  7. Variance filtering (bottom 20%)')
print('  8. Confounder check (PCA)')
print('  9. Age + sex regression')
print(' 10. INT normalisation (Blom)')
print(' 11. Split CD and UC')
print(' 12. Save all matrices')

IBD PROTEOMIC PREPROCESSING PIPELINE
Steps:
  1. Load data
  2. Remove IBD-unclassified
  3. Derive batch from missingness
  4. Protein QC (>30% missing → remove)
  5. Sample QC  (>30% missing → remove)
  6. Imputation comparison
     KNN vs Median vs MICE
     Select best via KS test
  7. Variance filtering (bottom 20%)
  8. Confounder check (PCA)
  9. Age + sex regression
 10. INT normalisation (Blom)
 11. Split CD and UC
 12. Save all matrices


In [2]:
# ============================================================
# STEP 1 — LOAD DATA
# ============================================================
print('\nSTEP 1 — LOAD DATA')
print('='*55)

prots = pd.read_csv(
    RAW_PROTEINS, sep='\t',
    compression='gzip',
    low_memory=False)

meta = pd.read_csv(
    RAW_METADATA, sep='\t',
    low_memory=False)

# Drop PC columns — not used in any analysis
pc_cols = [c for c in meta.columns
            if c.startswith('pc')]
meta = meta.drop(columns=pc_cols)

print(f'Protein matrix : {prots.shape}')
print(f'Metadata       : {meta.shape}')
print(f'Metadata cols  : {meta.columns.tolist()}')
print(f'\nDiagnosis counts:')
print(meta['diagnosis'].value_counts())
print(f'\nOverall missingness: '
      f'{prots.isna().mean().mean()*100:.2f}%')

# Missingness distribution
miss = prots.isna().mean()
print(f'\nProtein missingness distribution:')
for lo,hi,lab in [
    (0,    0.001,'0%      '),
    (0.001,0.10, '0.1-10% '),
    (0.10, 0.20, '10-20%  '),
    (0.20, 0.30, '20-30%  '),
    (0.30, 0.50, '30-50%  '),
    (0.50, 1.01, '>50%    '),
]:
    n = ((miss>=lo)&(miss<hi)).sum()
    print(f'  {lab}: {n} proteins')


STEP 1 — LOAD DATA
Protein matrix : (658, 2923)
Metadata       : (658, 9)
Metadata cols  : ['id', 'sex', 'age', 'bmi', 'smoking_status', 'smoking_past', 'hba1c', 'ibd_status', 'diagnosis']

Diagnosis counts:
diagnosis
UC     437
CD     217
IBD      4
Name: count, dtype: int64

Overall missingness: 10.42%

Protein missingness distribution:
  0%      : 0 proteins
  0.1-10% : 1458 proteins
  10-20%  : 1353 proteins
  20-30%  : 109 proteins
  30-50%  : 0 proteins
  >50%    : 3 proteins


In [3]:
# ============================================================
# STEP 2 — REMOVE IBD-UNCLASSIFIED
# ============================================================
print('\nSTEP 2 — REMOVE IBD-UNCLASSIFIED')
print('='*55)

n_before = len(meta)
keep  = meta['diagnosis'].isin(['CD','UC'])
meta  = meta[keep].reset_index(drop=True)
prots = prots[keep].reset_index(drop=True)

print(f'Removed: {n_before - len(meta)} patients')
print(f'Remaining: {len(meta)} patients')
print(meta['diagnosis'].value_counts())


STEP 2 — REMOVE IBD-UNCLASSIFIED
Removed: 4 patients
Remaining: 654 patients
diagnosis
UC    437
CD    217
Name: count, dtype: int64


In [4]:
# ============================================================
# STEP 3 — DERIVE BATCH FROM MISSINGNESS
# ============================================================
print('\nSTEP 3 — DERIVE BATCH')
print('='*55)

miss_per_prot = prots.isna().mean()

# Panel 2 proteins: 8-22% missing
# (missing only in Panel-1-only patients)
panel2 = miss_per_prot[
    (miss_per_prot >= 0.08) &
    (miss_per_prot <= 0.22)].index.tolist()

print(f'Panel 2 candidate proteins: {len(panel2)}')

# Patients with >80% of Panel2 missing
# = Panel-1-only patients
p2_miss = prots[panel2].isna().mean(axis=1)
batch   = (p2_miss > 0.80).astype(int)
meta['batch'] = batch.values

print(f'\nBatch distribution:')
print(pd.crosstab(meta['diagnosis'],
                   meta['batch']))
print(f'\nbatch=0: both panels')
print(f'batch=1: Panel 1 only')

# Save batch info
pd.DataFrame({
    'threshold': ['Panel2 missing > 80%'],
    'CD_batch0': [(meta[meta['diagnosis']=='CD']['batch']==0).sum()],
    'CD_batch1': [(meta[meta['diagnosis']=='CD']['batch']==1).sum()],
    'UC_batch0': [(meta[meta['diagnosis']=='UC']['batch']==0).sum()],
    'UC_batch1': [(meta[meta['diagnosis']=='UC']['batch']==1).sum()],
}).to_csv(f'{TABLES_DIR}/batch_derivation.csv',
           index=False)
print('✓ Saved: batch_derivation.csv')


STEP 3 — DERIVE BATCH
Panel 2 candidate proteins: 1462

Batch distribution:
batch        0   1
diagnosis         
CD         191  26
UC         364  73

batch=0: both panels
batch=1: Panel 1 only
✓ Saved: batch_derivation.csv


In [5]:
# ============================================================
# STEP 4 — PROTEIN QC (>30% missing → remove)
# ============================================================
print('\nSTEP 4 — PROTEIN QC (>30% missing)')
print('='*55)

miss_prot = prots.isna().mean()
n_before  = prots.shape[1]

# Remove >30% missing
keep_prot = miss_prot[miss_prot <= 0.30].index
prots     = prots[keep_prot]

n_removed = n_before - prots.shape[1]
print(f'Before : {n_before} proteins')
print(f'Removed: {n_removed} proteins '
      f'(>{30}% missing)')
print(f'After  : {prots.shape[1]} proteins')

# Breakdown of removed
print(f'\nBreakdown of removed proteins:')
removed_miss = miss_prot[miss_prot > 0.30]
for lo,hi,lab in [
    (0.30,0.50,'30-50%'),
    (0.50,1.01,'>50%  '),
]:
    n = ((removed_miss>=lo)&
          (removed_miss<hi)).sum()
    print(f'  {lab}: {n} proteins')


STEP 4 — PROTEIN QC (>30% missing)
Before : 2923 proteins
Removed: 3 proteins (>30% missing)
After  : 2920 proteins

Breakdown of removed proteins:
  30-50%: 0 proteins
  >50%  : 3 proteins


In [6]:
# ============================================================
# STEP 5 — SAMPLE QC (>30% missing → remove)
# ============================================================
print('\nSTEP 5 — SAMPLE QC (>30% missing)')
print('='*55)

miss_samp = prots.isna().mean(axis=1)
n_before  = len(meta)

keep_samp = miss_samp[
    miss_samp <= 0.30].index
prots = prots.loc[keep_samp].reset_index(
    drop=True)
meta  = meta.loc[keep_samp].reset_index(
    drop=True)

n_removed = n_before - len(meta)
print(f'Before : {n_before} patients')
print(f'Removed: {n_removed} patients '
      f'(>{30}% missing)')
print(f'After  : {len(meta)} patients')
print(f'\nCD: {(meta["diagnosis"]=="CD").sum()}')
print(f'UC: {(meta["diagnosis"]=="UC").sum()}')
print(f'\nBatch after QC:')
print(pd.crosstab(meta['diagnosis'],
                   meta['batch']))
print(f'\nRemaining missingness: '
      f'{prots.isna().mean().mean()*100:.2f}%')


STEP 5 — SAMPLE QC (>30% missing)
Before : 654 patients
Removed: 105 patients (>30% missing)
After  : 549 patients

CD: 190
UC: 359

Batch after QC:
batch        0
diagnosis     
CD         190
UC         359

Remaining missingness: 2.61%


In [7]:
# ============================================================
# STEP 6 — IMPUTATION COMPARISON
# KNN vs Median vs MICE
# Select best via KS test
# ============================================================
print('\nSTEP 6 — IMPUTATION COMPARISON')
print('='*55)

# Select 200 random proteins with missing values
# for comparison
np.random.seed(SEED)
has_missing = prots.columns[
    prots.isna().any()].tolist()
test_prots  = np.random.choice(
    has_missing,
    size=min(200, len(has_missing)),
    replace=False)

print(f'Proteins with missing values: '
      f'{len(has_missing)}')
print(f'Test proteins for comparison: '
      f'{len(test_prots)}')

# ── Method 1: KNN imputation ──────────────────────
print('\nMethod 1: KNN (k=10)...')
imp_knn   = KNNImputer(n_neighbors=KNN_K)
X_knn     = pd.DataFrame(
    imp_knn.fit_transform(prots.values),
    columns=prots.columns)
print('  ✓ Done')

# ── Method 2: Median imputation ───────────────────
print('Method 2: Median...')
X_median  = prots.copy()
for col in prots.columns:
    med = prots[col].median()
    X_median[col] = prots[col].fillna(med)
print('  ✓ Done')

# ── Method 3: MICE (iterative imputer) ────────────
print('Method 3: MICE (IterativeImputer)...')
from sklearn.experimental import (
    enable_iterative_imputer)
from sklearn.impute import IterativeImputer

imp_mice  = IterativeImputer(
    max_iter=10, random_state=SEED,
    n_nearest_features=50)
X_mice    = pd.DataFrame(
    imp_mice.fit_transform(prots.values),
    columns=prots.columns)
print('  ✓ Done')


STEP 6 — IMPUTATION COMPARISON
Proteins with missing values: 2920
Test proteins for comparison: 200

Method 1: KNN (k=10)...
  ✓ Done
Method 2: Median...
  ✓ Done
Method 3: MICE (IterativeImputer)...
  ✓ Done


In [8]:
# ── KS test comparison ────────────────────────────
print('\nKS TEST COMPARISON')
print('='*55)
print('Comparing imputed distributions to')
print('observed distributions')
print('Lower KS statistic = better preservation')
print('of original distribution\n')

ks_knn    = []
ks_median = []
ks_mice   = []

for col in test_prots:
    # Observed values (non-missing)
    obs = prots[col].dropna().values

    if len(obs) < 10:
        continue

    # Imputed values for missing positions
    missing_mask = prots[col].isna()
    if missing_mask.sum() == 0:
        continue

    knn_imp    = X_knn.loc[
        missing_mask, col].values
    med_imp    = X_median.loc[
        missing_mask, col].values
    mice_imp   = X_mice.loc[
        missing_mask, col].values

    # KS test: imputed vs observed
    ks_knn.append(
        ks_2samp(obs, knn_imp).statistic)
    ks_median.append(
        ks_2samp(obs, med_imp).statistic)
    ks_mice.append(
        ks_2samp(obs, mice_imp).statistic)

ks_knn_mean    = np.mean(ks_knn)
ks_median_mean = np.mean(ks_median)
ks_mice_mean   = np.mean(ks_mice)

print(f'{"Method":<12} {"Mean KS":>10} '
      f'{"Median KS":>12} {"Std KS":>10}')
print('-'*46)
print(f'{"KNN":<12} {ks_knn_mean:>10.4f} '
      f'{np.median(ks_knn):>12.4f} '
      f'{np.std(ks_knn):>10.4f}')
print(f'{"Median":<12} {ks_median_mean:>10.4f} '
      f'{np.median(ks_median):>12.4f} '
      f'{np.std(ks_median):>10.4f}')
print(f'{"MICE":<12} {ks_mice_mean:>10.4f} '
      f'{np.median(ks_mice):>12.4f} '
      f'{np.std(ks_mice):>10.4f}')

# Select best method
best_method = min(
    [('KNN',    ks_knn_mean,    X_knn),
     ('Median', ks_median_mean, X_median),
     ('MICE',   ks_mice_mean,   X_mice)],
    key=lambda x: x[1])

print(f'\nBEST METHOD: {best_method[0]}')
print(f'(lowest mean KS = '
      f'best distribution preservation)')
X_imp = best_method[2].copy()

# Save KS results
pd.DataFrame({
    'method'   : ['KNN','Median','MICE'],
    'mean_ks'  : [ks_knn_mean,
                   ks_median_mean,
                   ks_mice_mean],
    'median_ks': [np.median(ks_knn),
                   np.median(ks_median),
                   np.median(ks_mice)],
    'std_ks'   : [np.std(ks_knn),
                   np.std(ks_median),
                   np.std(ks_mice)],
    'selected' : [
        best_method[0]=='KNN',
        best_method[0]=='Median',
        best_method[0]=='MICE']
}).to_csv(f'{TABLES_DIR}/imputation_ks_comparison.csv',
           index=False)
print('✓ Saved: imputation_ks_comparison.csv')


KS TEST COMPARISON
Comparing imputed distributions to
observed distributions
Lower KS statistic = better preservation
of original distribution

Method          Mean KS    Median KS     Std KS
----------------------------------------------
KNN              0.3156       0.3132     0.0909
Median           0.4994       0.5000     0.0008
MICE             0.3107       0.2940     0.1217

BEST METHOD: MICE
(lowest mean KS = best distribution preservation)
✓ Saved: imputation_ks_comparison.csv


In [9]:
# ── KS test figure ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor('white')

methods = [
    ('KNN',    ks_knn,    '#1B3A6B'),
    ('Median', ks_median, '#C96A1F'),
    ('MICE',   ks_mice,   '#2E7D5B'),
]
for ax, (name, ks_vals, col) in zip(
        axes, methods):
    ax.set_facecolor('#FAFAFA')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.hist(ks_vals, bins=30,
             color=col, alpha=0.85,
             edgecolor='white')
    ax.axvline(np.mean(ks_vals),
                color='#E24B4A',
                linewidth=2, linestyle='--',
                label=f'Mean={np.mean(ks_vals):.4f}')
    ax.set_xlabel('KS statistic', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title(
        f'{name} imputation\n'
        f'Mean KS={np.mean(ks_vals):.4f}',
        fontsize=12, fontweight='700',
        loc='left')
    ax.legend(fontsize=10)

plt.suptitle(
    'Imputation method comparison — KS test\n'
    'Lower KS = better distribution preservation',
    fontsize=13, fontweight='800', y=1.02)
plt.tight_layout()
plt.savefig(
    f'{FIGURES_DIR}/imputation_ks_comparison.png',
    dpi=200, bbox_inches='tight',
    facecolor='white')
plt.close()
print('✓ Saved: imputation_ks_comparison.png')

✓ Saved: imputation_ks_comparison.png


In [10]:
# ============================================================
# STEP 7 — VARIANCE FILTERING (bottom 20%)
# ============================================================
print('\nSTEP 7 — VARIANCE FILTERING')
print('='*55)

variances = X_imp.var(axis=0)
threshold = np.percentile(variances, 20)
keep_var  = variances[
    variances > threshold].index.tolist()

print(f'Before   : {X_imp.shape[1]} proteins')
print(f'Threshold: {threshold:.4f} '
      f'(20th percentile)')
print(f'Removed  : '
      f'{X_imp.shape[1]-len(keep_var)} proteins')
print(f'After    : {len(keep_var)} proteins')

X_var = X_imp[keep_var]

pd.Series(keep_var).to_csv(
    f'{DATA_DIR}/protein_cols_varfiltered.csv',
    index=False, header=False)
print('✓ Saved: protein_cols_varfiltered.csv')


STEP 7 — VARIANCE FILTERING
Before   : 2920 proteins
Threshold: 0.1361 (20th percentile)
Removed  : 584 proteins
After    : 2336 proteins
✓ Saved: protein_cols_varfiltered.csv


In [11]:
# ============================================================
# STEP 8 — CONFOUNDER CHECK (PCA)
# ============================================================
print('\nSTEP 8 — CONFOUNDER CHECK')
print('='*55)

pca   = PCA(n_components=10, random_state=SEED)
X_pca = pca.fit_transform(X_var.values)
var_e = pca.explained_variance_ratio_*100

print(f'Variance explained:')
for i in range(5):
    print(f'  PC{i+1}: {var_e[i]:.2f}%')

print(f'\n{"Covariate":<12} '
      f'{"PC1 r":>8} {"PC2 r":>8} '
      f'{"PC1 p":>10} {"PC2 p":>10}')
print('-'*52)

covariates = {}
covariates['age']   = meta['age'].values
covariates['sex']   = meta['sex'].values
covariates['batch'] = meta['batch'].values

confounder_results = []
for name, vals in covariates.items():
    r1,p1 = stats.pearsonr(vals, X_pca[:,0])
    r2,p2 = stats.pearsonr(vals, X_pca[:,1])
    sig1  = '**' if p1<0.01 else (
            '*' if p1<0.05 else 'ns')
    sig2  = '**' if p2<0.01 else (
            '*' if p2<0.05 else 'ns')
    print(f'{name:<12} '
          f'{r1:>8.3f} {r2:>8.3f} '
          f'{p1:>10.4f}{sig1} '
          f'{p2:>10.4f}{sig2}')
    confounder_results.append({
        'covariate': name,
        'r_PC1': round(r1,4),
        'r_PC2': round(r2,4),
        'p_PC1': round(p1,4),
        'p_PC2': round(p2,4),
        'sig_PC2': sig2})

pd.DataFrame(confounder_results).to_csv(
    f'{TABLES_DIR}/confounder_check.csv',
    index=False)

# PCA figure before regression
fig, axes = plt.subplots(
    1, 3, figsize=(18, 6))
fig.patch.set_facecolor('white')

plot_configs = [
    ('Age',   meta['age'].values,
     'RdYlBu_r', False),
    ('Sex',   meta['sex'].values,
     None, True),
    ('Batch', meta['batch'].values,
     None, True),
]
for ax, (name, vals, cmap, cat) in zip(
        axes, plot_configs):
    ax.set_facecolor('#F8F9FA')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if cat:
        cols = ['#1B3A6B','#E24B4A']
        labs = (['Male','Female']
                 if name=='Sex'
                 else ['Both panels',
                       'Panel 1 only'])
        for i, uv in enumerate(
                np.unique(vals)):
            m = vals==uv
            ax.scatter(
                X_pca[m,0], X_pca[m,1],
                c=cols[i], s=20, alpha=0.7,
                linewidths=0,
                label=labs[i], zorder=3)
        ax.legend(fontsize=9, title=name)
    else:
        sc = ax.scatter(
            X_pca[:,0], X_pca[:,1],
            c=vals, cmap=cmap,
            s=20, alpha=0.7,
            linewidths=0, zorder=3)
        plt.colorbar(sc, ax=ax,
                      label=name, shrink=0.8)
    r1,_ = stats.pearsonr(vals, X_pca[:,0])
    r2,_ = stats.pearsonr(vals, X_pca[:,1])
    ax.set_title(
        f'{name}\n'
        f'r(PC1)={r1:.3f} r(PC2)={r2:.3f}',
        fontsize=11, fontweight='700',
        loc='left')
    ax.set_xlabel(
        f'PC1 ({var_e[0]:.1f}%)', fontsize=11)
    ax.set_ylabel(
        f'PC2 ({var_e[1]:.1f}%)', fontsize=11)

plt.suptitle(
    'PCA confounder check — before regression\n'
    f'n={len(meta)} IBD patients | '
    f'{len(keep_var)} proteins',
    fontsize=13, fontweight='800', y=1.02)
plt.tight_layout()
plt.savefig(
    f'{FIGURES_DIR}/pca_before_correction.png',
    dpi=200, bbox_inches='tight',
    facecolor='white')
plt.close()
print('✓ Saved: pca_before_correction.png')


STEP 8 — CONFOUNDER CHECK
Variance explained:
  PC1: 26.04%
  PC2: 4.23%
  PC3: 2.73%
  PC4: 2.38%
  PC5: 1.94%

Covariate       PC1 r    PC2 r      PC1 p      PC2 p
----------------------------------------------------
age            -0.008    0.223     0.8532ns     0.0000**
sex            -0.038    0.365     0.3725ns     0.0000**
batch             nan      nan        nanns        nanns
✓ Saved: pca_before_correction.png


In [12]:
# ============================================================
# STEP 9 — AGE AND SEX REGRESSION
# ============================================================
print('\nSTEP 9 — AGE AND SEX REGRESSION')
print('='*55)

def regress_covariate(X_arr, covariate,
                       cov_name):
    X_res  = X_arr.copy()
    r2_arr = np.zeros(X_arr.shape[1])
    for j in range(X_arr.shape[1]):
        y  = X_arr[:,j]
        sl, ic, r, _, _ = stats.linregress(
            covariate, y)
        X_res[:,j] = y - (ic + sl*covariate)
        r2_arr[j]  = r**2
    print(f'{cov_name}: mean R²='
          f'{r2_arr.mean():.4f} '
          f'max={r2_arr.max():.4f} '
          f'R²>0.10: {(r2_arr>0.1).sum()}')
    return X_res, r2_arr

# Step 9a: Age regression
print('Regressing age...')
X_res, r2_age = regress_covariate(
    X_var.values,
    meta['age'].values, 'Age')

# Step 9b: Sex regression
print('Regressing sex...')
X_res, r2_sex = regress_covariate(
    X_res,
    meta['sex'].values, 'Sex')

# Verify both removed
pca2  = PCA(n_components=2, random_state=SEED)
Xp2   = pca2.fit_transform(X_res)
var2  = pca2.explained_variance_ratio_*100

print(f'\nVerification:')
print(f'{"Covariate":<10} {"Before":>8} '
      f'{"After":>8}')
print('-'*30)
for name, vals, r_before in [
    ('Age',   meta['age'].values,
     confounder_results[0]['r_PC2']),
    ('Sex',   meta['sex'].values,
     confounder_results[1]['r_PC2']),
    ('Batch', meta['batch'].values,
     confounder_results[2]['r_PC2']),
]:
    r_after, p_after = stats.pearsonr(
        vals, Xp2[:,1])
    print(f'{name:<10} {r_before:>8.4f} '
          f'{r_after:>8.4f}')

# PCA after regression
fig, axes = plt.subplots(
    1, 3, figsize=(18, 6))
fig.patch.set_facecolor('white')

for ax, (name, vals, cmap, cat) in zip(
        axes, plot_configs):
    ax.set_facecolor('#F8F9FA')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if cat:
        cols = ['#1B3A6B','#E24B4A']
        labs = (['Male','Female']
                 if name=='Sex'
                 else ['Both panels',
                       'Panel 1 only'])
        for i, uv in enumerate(
                np.unique(vals)):
            m = vals==uv
            ax.scatter(
                Xp2[m,0], Xp2[m,1],
                c=cols[i], s=20, alpha=0.7,
                linewidths=0,
                label=labs[i], zorder=3)
        ax.legend(fontsize=9, title=name)
    else:
        sc = ax.scatter(
            Xp2[:,0], Xp2[:,1],
            c=vals, cmap=cmap,
            s=20, alpha=0.7,
            linewidths=0, zorder=3)
        plt.colorbar(sc, ax=ax,
                      label=name, shrink=0.8)
    r1,_ = stats.pearsonr(vals, Xp2[:,0])
    r2,_ = stats.pearsonr(vals, Xp2[:,1])
    ax.set_title(
        f'{name}\n'
        f'r(PC1)={r1:.3f} r(PC2)={r2:.3f}',
        fontsize=11, fontweight='700',
        loc='left')
    ax.set_xlabel(
        f'PC1 ({var2[0]:.1f}%)', fontsize=11)
    ax.set_ylabel(
        f'PC2 ({var2[1]:.1f}%)', fontsize=11)

plt.suptitle(
    'PCA after age + sex regression\n'
    f'n={len(meta)} IBD patients | '
    f'{len(keep_var)} proteins',
    fontsize=13, fontweight='800', y=1.02)
plt.tight_layout()
plt.savefig(
    f'{FIGURES_DIR}/pca_after_correction.png',
    dpi=200, bbox_inches='tight',
    facecolor='white')
plt.close()
print('✓ Saved: pca_after_correction.png')


STEP 9 — AGE AND SEX REGRESSION
Regressing age...
Age: mean R²=0.0091 max=0.3406 R²>0.10: 25
Regressing sex...
Sex: mean R²=0.0151 max=0.7355 R²>0.10: 53

Verification:
Covariate    Before    After
------------------------------
Age          0.2229   0.0018
Sex          0.3650  -0.0000
Batch           nan      nan
✓ Saved: pca_after_correction.png


In [13]:
# ============================================================
# STEP 10 — INT NORMALISATION (Blom formula)
# ============================================================
print('\nSTEP 10 — INT NORMALISATION')
print('='*55)

n     = X_res.shape[0]
X_int = X_res.copy()

for j in range(X_res.shape[1]):
    ranks      = pd.Series(
        X_res[:,j]).rank()
    X_int[:,j] = norm.ppf(
        (ranks-0.375)/(n+0.25))

print(f'Mean: {X_int.mean():.6f}')
print(f'Std : {X_int.std():.6f}')
print(f'NaN : {np.isnan(X_int).any()}')

# limma matrix = imputed only (no regression)
# age, sex, batch as covariates in limma model
X_limma = X_var.values.copy()


STEP 10 — INT NORMALISATION
Mean: 0.000000
Std : 0.996410
NaN : False


In [14]:
# ============================================================
# STEP 11 — SPLIT CD AND UC
# ============================================================
print('\nSTEP 11 — SPLIT CD AND UC')
print('='*55)

cd = (meta['diagnosis']=='CD').values
uc = (meta['diagnosis']=='UC').values

meta_cd = meta[cd].reset_index(drop=True)
meta_uc = meta[uc].reset_index(drop=True)

X_cd_int   = X_int[cd]
X_uc_int   = X_int[uc]
X_cd_limma = X_limma[cd]
X_uc_limma = X_limma[uc]

print(f'CD clustering matrix : {X_cd_int.shape}')
print(f'UC clustering matrix : {X_uc_int.shape}')
print(f'\nCD batch:')
print(meta_cd['batch'].value_counts())
print(f'\nUC batch:')
print(meta_uc['batch'].value_counts())
print(f'\nCD sex: '
      f'male={(meta_cd["sex"]==0).sum()} '
      f'female={(meta_cd["sex"]==1).sum()}')
print(f'UC sex: '
      f'male={(meta_uc["sex"]==0).sum()} '
      f'female={(meta_uc["sex"]==1).sum()}')
print(f'CD age: '
      f'mean={meta_cd["age"].mean():.1f} '
      f'sd={meta_cd["age"].std():.1f}')
print(f'UC age: '
      f'mean={meta_uc["age"].mean():.1f} '
      f'sd={meta_uc["age"].std():.1f}')


STEP 11 — SPLIT CD AND UC
CD clustering matrix : (190, 2336)
UC clustering matrix : (359, 2336)

CD batch:
batch
0    190
Name: count, dtype: int64

UC batch:
batch
0    359
Name: count, dtype: int64

CD sex: male=99 female=91
UC sex: male=178 female=181
CD age: mean=56.0 sd=8.3
UC age: mean=58.7 sd=7.7


In [15]:
# ============================================================
# STEP 12 — SAVE ALL MATRICES
# ============================================================
print('\nSTEP 12 — SAVE ALL MATRICES')
print('='*55)

np.save(f'{DATA_DIR}/X_cd_int.npy',   X_cd_int)
np.save(f'{DATA_DIR}/X_uc_int.npy',   X_uc_int)
np.save(f'{DATA_DIR}/X_cd_limma.npy', X_cd_limma)
np.save(f'{DATA_DIR}/X_uc_limma.npy', X_uc_limma)

meta_cd.to_csv(
    f'{DATA_DIR}/meta_cd_sampleqc.csv',
    index=False)
meta_uc.to_csv(
    f'{DATA_DIR}/meta_uc_sampleqc.csv',
    index=False)
meta.to_csv(
    f'{DATA_DIR}/meta_all_postqc.csv',
    index=False)
pd.Series(keep_var).to_csv(
    f'{DATA_DIR}/protein_cols_varfiltered.csv',
    index=False, header=False)

print('Files saved:')
saved = [
    'X_cd_int.npy','X_uc_int.npy',
    'X_cd_limma.npy','X_uc_limma.npy',
    'meta_cd_sampleqc.csv',
    'meta_uc_sampleqc.csv',
    'meta_all_postqc.csv',
    'protein_cols_varfiltered.csv']
for f in saved:
    size = os.path.getsize(
        f'{DATA_DIR}/{f}')//1024
    print(f'  ✅ {f} ({size} KB)')

print(f'\n{"="*55}')
print(f'PREPROCESSING COMPLETE')
print(f'{"="*55}')
print(f'Patients: {len(meta)} total')
print(f'  CD: {len(meta_cd)}')
print(f'  UC: {len(meta_uc)}')
print(f'Proteins: {len(keep_var)}')
print(f'\nImputation method: {best_method[0]}')
print(f'  (lowest KS={best_method[1]:.4f})')
print(f'\nCovariates regressed:')
print(f'  Age | Sex')
print(f'\nCovariates in limma:')
print(f'  Age + Sex + Batch')
print(f'\nNext: 02_traditional_clustering.py')


STEP 12 — SAVE ALL MATRICES
Files saved:
  ✅ X_cd_int.npy (3467 KB)
  ✅ X_uc_int.npy (6551 KB)
  ✅ X_cd_limma.npy (3467 KB)
  ✅ X_uc_limma.npy (6551 KB)
  ✅ meta_cd_sampleqc.csv (10 KB)
  ✅ meta_uc_sampleqc.csv (21 KB)
  ✅ meta_all_postqc.csv (31 KB)
  ✅ protein_cols_varfiltered.csv (13 KB)

PREPROCESSING COMPLETE
Patients: 549 total
  CD: 190
  UC: 359
Proteins: 2336

Imputation method: MICE
  (lowest KS=0.3107)

Covariates regressed:
  Age | Sex

Covariates in limma:
  Age + Sex + Batch

Next: 02_traditional_clustering.py


In [16]:
import numpy as np
import pandas as pd
from sklearn.experimental import (
    enable_iterative_imputer)
from sklearn.impute import (
    KNNImputer, IterativeImputer)
from sklearn.decomposition import PCA
from scipy import stats
from scipy.stats import norm, ks_2samp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import os

PROJECT_DIR  = '/rds/homes/j/jxt554'
DATA_DIR     = f'{PROJECT_DIR}/data'
FIGURES_DIR  = f'{PROJECT_DIR}/figures'
TABLES_DIR   = f'{PROJECT_DIR}/tables'
SEED         = 42
KNN_K        = 10

RAW_PROTEINS = ('/rds/projects/a/acharjea-multi-omics'
                '/proteomics/ukbb_ibd_proteins.tsv.gz')
RAW_METADATA = ('/rds/projects/a/acharjea-multi-omics'
                '/proteomics/ukbb_ibd_metadata.tsv')

for d in [DATA_DIR, FIGURES_DIR, TABLES_DIR]:
    os.makedirs(d, exist_ok=True)

print('IBD PREPROCESSING — PANEL 1 APPROACH')
print('='*55)

prots = pd.read_csv(
    RAW_PROTEINS, sep='\t',
    compression='gzip', low_memory=False)
meta  = pd.read_csv(
    RAW_METADATA, sep='\t', low_memory=False)

pc_cols = [c for c in meta.columns
            if c.startswith('pc')]
meta = meta.drop(columns=pc_cols)

print(f'Raw: {prots.shape}')
print(meta['diagnosis'].value_counts())

IBD PREPROCESSING — PANEL 1 APPROACH
Raw: (658, 2923)
diagnosis
UC     437
CD     217
IBD      4
Name: count, dtype: int64


In [17]:
# ── Step 1: Remove IBD-unclassified ───────────────
print('\nSTEP 1 — REMOVE IBD-UNCLASSIFIED')
print('='*55)
keep  = meta['diagnosis'].isin(['CD','UC'])
meta  = meta[keep].reset_index(drop=True)
prots = prots[keep].reset_index(drop=True)
print(f'Remaining: {len(meta)} patients')
print(meta['diagnosis'].value_counts())

# ── Step 2: Derive batch ──────────────────────────
print('\nSTEP 2 — DERIVE BATCH')
print('='*55)

miss_per_prot = prots.isna().mean()

panel2 = miss_per_prot[
    (miss_per_prot >= 0.08) &
    (miss_per_prot <= 0.22)].index.tolist()
print(f'Panel 2 candidates: {len(panel2)}')

p2_miss = prots[panel2].isna().mean(axis=1)
batch   = (p2_miss > 0.80).astype(int)
meta['batch'] = batch.values

print('Batch by diagnosis:')
print(pd.crosstab(meta['diagnosis'],
                   meta['batch']))

pd.DataFrame({
    'CD_batch0':[(meta[meta['diagnosis']=='CD']['batch']==0).sum()],
    'CD_batch1':[(meta[meta['diagnosis']=='CD']['batch']==1).sum()],
    'UC_batch0':[(meta[meta['diagnosis']=='UC']['batch']==0).sum()],
    'UC_batch1':[(meta[meta['diagnosis']=='UC']['batch']==1).sum()],
}).to_csv(f'{TABLES_DIR}/batch_derivation.csv',
           index=False)
print('✓ Saved: batch_derivation.csv')

# ── Step 3: Protein QC — remove >50% missing ──────
print('\nSTEP 3 — PROTEIN QC (>50% missing)')
print('='*55)
miss     = prots.isna().mean()
n_before = prots.shape[1]
prots    = prots.loc[:, miss <= 0.50]
print(f'Removed >50% missing: '
      f'{n_before-prots.shape[1]} proteins')
print(f'Remaining: {prots.shape[1]} proteins')

# ── Step 4: Panel 1 decision ──────────────────────
print('\nSTEP 4 — PANEL 1 DECISION')
print('='*55)
print('Keeping Panel 1 proteins only')
print('(<5% missing = present in all patients)')

miss = prots.isna().mean()
panel1_prots = miss[miss < 0.05].index.tolist()
print(f'Panel 1 proteins: {len(panel1_prots)}')

prots = prots[panel1_prots]
print(f'Matrix after panel decision: '
      f'{prots.shape}')
print(f'Remaining missingness: '
      f'{prots.isna().mean().mean()*100:.2f}%')


STEP 1 — REMOVE IBD-UNCLASSIFIED
Remaining: 654 patients
diagnosis
UC    437
CD    217
Name: count, dtype: int64

STEP 2 — DERIVE BATCH
Panel 2 candidates: 1462
Batch by diagnosis:
batch        0   1
diagnosis         
CD         191  26
UC         364  73
✓ Saved: batch_derivation.csv

STEP 3 — PROTEIN QC (>50% missing)
Removed >50% missing: 3 proteins
Remaining: 2920 proteins

STEP 4 — PANEL 1 DECISION
Keeping Panel 1 proteins only
(<5% missing = present in all patients)
Panel 1 proteins: 1239
Matrix after panel decision: (654, 1239)
Remaining missingness: 2.45%


In [18]:
# ── Step 5: Sample QC ─────────────────────────────
print('\nSTEP 5 — SAMPLE QC (>30% missing)')
print('='*55)
miss_s    = prots.isna().mean(axis=1)
n_before  = len(meta)
keep_s    = miss_s[miss_s <= 0.30].index
prots     = prots.loc[keep_s].reset_index(
    drop=True)
meta      = meta.loc[keep_s].reset_index(
    drop=True)
print(f'Removed: {n_before-len(meta)} patients')
print(f'CD: {(meta["diagnosis"]=="CD").sum()}')
print(f'UC: {(meta["diagnosis"]=="UC").sum()}')
print(f'\nBatch after QC:')
print(pd.crosstab(meta['diagnosis'],
                   meta['batch']))
print(f'Remaining missingness: '
      f'{prots.isna().mean().mean()*100:.2f}%')

# ── Step 6: Imputation comparison ─────────────────
print('\nSTEP 6 — IMPUTATION COMPARISON')
print('='*55)

np.random.seed(SEED)
has_miss   = prots.columns[
    prots.isna().any()].tolist()
test_prots = np.random.choice(
    has_miss,
    size=min(200, len(has_miss)),
    replace=False)

print(f'Proteins with missing: {len(has_miss)}')
print(f'Test set size: {len(test_prots)}')

print('\nKNN imputation...')
imp_knn  = KNNImputer(n_neighbors=KNN_K)
X_knn    = pd.DataFrame(
    imp_knn.fit_transform(prots.values),
    columns=prots.columns)

print('Median imputation...')
X_median = prots.copy()
for col in prots.columns:
    X_median[col] = prots[col].fillna(
        prots[col].median())

print('MICE imputation...')
imp_mice = IterativeImputer(
    max_iter=10, random_state=SEED,
    n_nearest_features=50)
X_mice   = pd.DataFrame(
    imp_mice.fit_transform(prots.values),
    columns=prots.columns)

# KS test
ks_knn, ks_med, ks_mic = [], [], []
for col in test_prots:
    obs  = prots[col].dropna().values
    mask = prots[col].isna()
    if len(obs)<10 or mask.sum()==0:
        continue
    ks_knn.append(ks_2samp(
        obs, X_knn.loc[mask,col].values
        ).statistic)
    ks_med.append(ks_2samp(
        obs, X_median.loc[mask,col].values
        ).statistic)
    ks_mic.append(ks_2samp(
        obs, X_mice.loc[mask,col].values
        ).statistic)

print(f'\n{"Method":<10} {"Mean KS":>10} '
      f'{"Median KS":>12}')
print('-'*35)
print(f'{"KNN":<10} {np.mean(ks_knn):>10.4f} '
      f'{np.median(ks_knn):>12.4f}')
print(f'{"Median":<10} {np.mean(ks_med):>10.4f} '
      f'{np.median(ks_med):>12.4f}')
print(f'{"MICE":<10} {np.mean(ks_mic):>10.4f} '
      f'{np.median(ks_mic):>12.4f}')

best = min(
    [('KNN',    np.mean(ks_knn), X_knn),
     ('Median', np.mean(ks_med), X_median),
     ('MICE',   np.mean(ks_mic), X_mice)],
    key=lambda x: x[1])

print(f'\nBEST: {best[0]} '
      f'(KS={best[1]:.4f})')
X_imp = best[2].copy()

pd.DataFrame({
    'method'  :['KNN','Median','MICE'],
    'mean_ks' :[np.mean(ks_knn),
                np.mean(ks_med),
                np.mean(ks_mic)],
    'selected':[best[0]=='KNN',
                best[0]=='Median',
                best[0]=='MICE']
}).to_csv(
    f'{TABLES_DIR}/imputation_ks_comparison.csv',
    index=False)


STEP 5 — SAMPLE QC (>30% missing)
Removed: 9 patients
CD: 215
UC: 430

Batch after QC:
batch        0   1
diagnosis         
CD         189  26
UC         358  72
Remaining missingness: 1.55%

STEP 6 — IMPUTATION COMPARISON
Proteins with missing: 1239
Test set size: 200

KNN imputation...
Median imputation...
MICE imputation...

Method        Mean KS    Median KS
-----------------------------------
KNN            0.3380       0.3413
Median         0.4998       0.5000
MICE           0.3300       0.3263

BEST: MICE (KS=0.3300)


In [19]:
# ── Step 7: Variance filtering ────────────────────
print('\nSTEP 7 — VARIANCE FILTERING (bottom 20%)')
print('='*55)

variances = X_imp.var(axis=0)
threshold = np.percentile(variances, 20)
keep_var  = variances[
    variances > threshold].index.tolist()
X_var     = X_imp[keep_var]

print(f'Before   : {X_imp.shape[1]} proteins')
print(f'After    : {len(keep_var)} proteins')
print(f'Removed  : '
      f'{X_imp.shape[1]-len(keep_var)}')

pd.Series(keep_var).to_csv(
    f'{DATA_DIR}/protein_cols_varfiltered.csv',
    index=False, header=False)

# ── Step 8: Confounder check ──────────────────────
print('\nSTEP 8 — CONFOUNDER CHECK (PCA)')
print('='*55)

pca   = PCA(n_components=10, random_state=SEED)
X_pca = pca.fit_transform(X_var.values)
var_e = pca.explained_variance_ratio_*100

print(f'PC1: {var_e[0]:.2f}%  '
      f'PC2: {var_e[1]:.2f}%')
print(f'\n{"Covariate":<10} '
      f'{"PC1 r":>8} {"PC2 r":>8} '
      f'{"PC2 p":>10} {"Sig":>5}')
print('-'*44)

conf_res = {}
for name, vals in {
    'age'  : meta['age'].values,
    'sex'  : meta['sex'].values,
    'batch': meta['batch'].values,
}.items():
    if np.std(vals) == 0:
        print(f'{name:<10} {"N/A":>8} '
              f'{"N/A":>8} {"N/A":>10} '
              f'{"N/A":>5}')
        conf_res[name] = (0, 1)
        continue
    r1,p1 = stats.pearsonr(vals, X_pca[:,0])
    r2,p2 = stats.pearsonr(vals, X_pca[:,1])
    sig   = '**' if p2<0.01 else (
            '*' if p2<0.05 else 'ns')
    print(f'{name:<10} {r1:>8.3f} '
          f'{r2:>8.3f} {p2:>10.4f} {sig:>5}')
    conf_res[name] = (r2, p2)


STEP 7 — VARIANCE FILTERING (bottom 20%)
Before   : 1239 proteins
After    : 991 proteins
Removed  : 248

STEP 8 — CONFOUNDER CHECK (PCA)
PC1: 27.01%  PC2: 6.00%

Covariate     PC1 r    PC2 r      PC2 p   Sig
--------------------------------------------
age           0.004    0.296     0.0000    **
sex          -0.006    0.190     0.0000    **
batch        -0.010    0.028     0.4716    ns


In [20]:
# ── Step 9: Age and sex regression ────────────────
print('\nSTEP 9 — AGE AND SEX REGRESSION')
print('='*55)

X_res = X_var.values.copy()

for cov_name, cov_vals in [
    ('Age', meta['age'].values),
    ('Sex', meta['sex'].values)]:
    r2_arr = np.zeros(X_res.shape[1])
    for j in range(X_res.shape[1]):
        y  = X_res[:,j]
        sl, ic, r, _, _ = stats.linregress(
            cov_vals, y)
        X_res[:,j] = y - (ic + sl*cov_vals)
        r2_arr[j]  = r**2
    pca2  = PCA(n_components=2,
                 random_state=SEED)
    Xp2   = pca2.fit_transform(X_res)
    r_aft, p_aft = stats.pearsonr(
        cov_vals, Xp2[:,1])
    r_bef = conf_res.get(
        cov_name.lower(), (0,1))[0]
    print(f'{cov_name}: '
          f'r before={r_bef:.4f} '
          f'→ after={r_aft:.4f} '
          f'p={p_aft:.4f}')

# ── Step 10: INT normalisation ────────────────────
print('\nSTEP 10 — INT NORMALISATION')
print('='*55)

n     = X_res.shape[0]
X_int = X_res.copy()
for j in range(X_res.shape[1]):
    ranks      = pd.Series(X_res[:,j]).rank()
    X_int[:,j] = norm.ppf(
        (ranks-0.375)/(n+0.25))

print(f'Mean: {X_int.mean():.6f}')
print(f'Std : {X_int.std():.6f}')
print(f'NaN : {np.isnan(X_int).any()}')

X_limma = X_var.values.copy()

# ── Step 11: Split CD and UC ──────────────────────
print('\nSTEP 11 — SPLIT CD AND UC')
print('='*55)

cd = meta['diagnosis']=='CD'
uc = meta['diagnosis']=='UC'

meta_cd = meta[cd].reset_index(drop=True)
meta_uc = meta[uc].reset_index(drop=True)
X_cd_int   = X_int[cd]
X_uc_int   = X_int[uc]
X_cd_limma = X_limma[cd]
X_uc_limma = X_limma[uc]

print(f'CD: {X_cd_int.shape}')
print(f'UC: {X_uc_int.shape}')
print(f'\nCD batch:')
print(meta_cd['batch'].value_counts())
print(f'\nUC batch:')
print(meta_uc['batch'].value_counts())
print(f'\nCD age: {meta_cd["age"].mean():.1f} '
      f'± {meta_cd["age"].std():.1f}')
print(f'UC age: {meta_uc["age"].mean():.1f} '
      f'± {meta_uc["age"].std():.1f}')

# ── Step 12: Save ─────────────────────────────────
print('\nSTEP 12 — SAVE')
print('='*55)

np.save(f'{DATA_DIR}/X_cd_int.npy',
         X_cd_int)
np.save(f'{DATA_DIR}/X_uc_int.npy',
         X_uc_int)
np.save(f'{DATA_DIR}/X_cd_limma.npy',
         X_cd_limma)
np.save(f'{DATA_DIR}/X_uc_limma.npy',
         X_uc_limma)

meta_cd.to_csv(
    f'{DATA_DIR}/meta_cd_sampleqc.csv',
    index=False)
meta_uc.to_csv(
    f'{DATA_DIR}/meta_uc_sampleqc.csv',
    index=False)
meta.to_csv(
    f'{DATA_DIR}/meta_all_postqc.csv',
    index=False)
pd.Series(keep_var).to_csv(
    f'{DATA_DIR}/protein_cols_varfiltered.csv',
    index=False, header=False)

print('✓ All files saved')
print('\nFINAL SUMMARY:')
print(f'CD: {X_cd_int.shape}')
print(f'UC: {X_uc_int.shape}')
print(f'Proteins: {len(keep_var)}')
print(f'\nImputation: {best[0]} '
      f'(KS={best[1]:.4f})')
print('\nPREPROCESSING COMPLETE ✓')
print('Next: 02_traditional_clustering.py')


STEP 9 — AGE AND SEX REGRESSION
Age: r before=0.2964 → after=-0.0000 p=1.0000
Sex: r before=0.1902 → after=0.0000 p=1.0000

STEP 10 — INT NORMALISATION
Mean: 0.000000
Std : 0.996886
NaN : False

STEP 11 — SPLIT CD AND UC
CD: (215, 991)
UC: (430, 991)

CD batch:
batch
0    189
1     26
Name: count, dtype: int64

UC batch:
batch
0    358
1     72
Name: count, dtype: int64

CD age: 56.2 ± 8.2
UC age: 58.7 ± 7.7

STEP 12 — SAVE
✓ All files saved

FINAL SUMMARY:
CD: (215, 991)
UC: (430, 991)
Proteins: 991

Imputation: MICE (KS=0.3300)

PREPROCESSING COMPLETE ✓
Next: 02_traditional_clustering.py


In [21]:
# ============================================================
# FINAL SUMMARY CHECK
# ============================================================
import numpy as np
import pandas as pd
import os

PROJECT_DIR = '/rds/homes/j/jxt554'
DATA_DIR    = f'{PROJECT_DIR}/data'
TABLES_DIR  = f'{PROJECT_DIR}/tables'

print('PREPROCESSING FINAL SUMMARY')
print('='*55)

# ── Matrices ──────────────────────────────────────
print('\nMATRICES:')
files = {
    'X_cd_int.npy'   : 'CD clustering (INT)',
    'X_uc_int.npy'   : 'UC clustering (INT)',
    'X_cd_limma.npy' : 'CD limma (raw)',
    'X_uc_limma.npy' : 'UC limma (raw)',
}
for fname, label in files.items():
    path = f'{DATA_DIR}/{fname}'
    if os.path.exists(path):
        X    = np.load(path)
        size = os.path.getsize(path)//1024
        print(f'  ✅ {label}: {X.shape} '
              f'({size} KB)')
    else:
        print(f'  ❌ {fname} NOT FOUND')

# ── Metadata ──────────────────────────────────────
print('\nMETADATA:')
meta_cd = pd.read_csv(
    f'{DATA_DIR}/meta_cd_sampleqc.csv')
meta_uc = pd.read_csv(
    f'{DATA_DIR}/meta_uc_sampleqc.csv')

print(f'\nCD (n={len(meta_cd)}):')
print(f'  Age  : {meta_cd["age"].mean():.1f} '
      f'± {meta_cd["age"].std():.1f}')
print(f'  Sex  : male={(meta_cd["sex"]==0).sum()} '
      f'female={(meta_cd["sex"]==1).sum()}')
print(f'  Batch: '
      f'0={(meta_cd["batch"]==0).sum()} '
      f'1={(meta_cd["batch"]==1).sum()}')

print(f'\nUC (n={len(meta_uc)}):')
print(f'  Age  : {meta_uc["age"].mean():.1f} '
      f'± {meta_uc["age"].std():.1f}')
print(f'  Sex  : male={(meta_uc["sex"]==0).sum()} '
      f'female={(meta_uc["sex"]==1).sum()}')
print(f'  Batch: '
      f'0={(meta_uc["batch"]==0).sum()} '
      f'1={(meta_uc["batch"]==1).sum()}')

# ── Proteins ──────────────────────────────────────
print('\nPROTEINS:')
prots = pd.read_csv(
    f'{DATA_DIR}/protein_cols_varfiltered.csv',
    header=None)[0].tolist()
print(f'  Final protein count: {len(prots)}')
print(f'  Sample proteins: '
      f'{prots[:3]} ... {prots[-3:]}')

# ── Imputation ────────────────────────────────────
print('\nIMPUTATION:')
ks_path = (f'{TABLES_DIR}/'
            f'imputation_ks_comparison.csv')
if os.path.exists(ks_path):
    ks_df = pd.read_csv(ks_path)
    print(ks_df.to_string(index=False))
    best = ks_df.loc[
        ks_df['mean_ks'].idxmin(), 'method']
    print(f'  Selected: {best}')

# ── Batch derivation ──────────────────────────────
print('\nBATCH DERIVATION:')
bat_path = f'{TABLES_DIR}/batch_derivation.csv'
if os.path.exists(bat_path):
    bat = pd.read_csv(bat_path)
    print(bat.to_string(index=False))

# ── Quality checks ────────────────────────────────
print('\nQUALITY CHECKS:')
X_cd = np.load(f'{DATA_DIR}/X_cd_int.npy')
X_uc = np.load(f'{DATA_DIR}/X_uc_int.npy')

print(f'  CD NaN: {np.isnan(X_cd).any()}')
print(f'  UC NaN: {np.isnan(X_uc).any()}')
print(f'  CD mean: {X_cd.mean():.6f} '
      f'(expect ~0)')
print(f'  CD std : {X_cd.std():.6f} '
      f'(expect ~1)')
print(f'  UC mean: {X_uc.mean():.6f}')
print(f'  UC std : {X_uc.std():.6f}')

# ── Pipeline summary ──────────────────────────────
print('\nPIPELINE STEPS COMPLETED:')
steps = [
    ('Load data',          '658 × 2923'),
    ('Remove IBD-unclass', '654 patients'),
    ('Derive batch',       'from missingness'),
    ('Protein QC >50%',    '3 removed'),
    ('Panel 1 decision',   '<5% missing kept'),
    ('Sample QC >30%',     'see counts below'),
    ('Imputation',         'best of KNN/Med/MICE'),
    ('Variance filter',    'bottom 20% removed'),
    ('Confounder check',   'PCA age/sex/batch'),
    ('Age regression',     'r → ~0'),
    ('Sex regression',     'r → ~0'),
    ('INT normalisation',  'Blom formula'),
    ('Split CD/UC',        'separate matrices'),
    ('Save all',           'data/ directory'),
]
for step, detail in steps:
    print(f'  ✅ {step:<22}: {detail}')

print('\n' + '='*55)
print('READY FOR CLUSTERING')
print('Next: 02_traditional_clustering.py')
print('='*55)

PREPROCESSING FINAL SUMMARY

MATRICES:
  ✅ CD clustering (INT): (215, 991) (1664 KB)
  ✅ UC clustering (INT): (430, 991) (3329 KB)
  ✅ CD limma (raw): (215, 991) (1664 KB)
  ✅ UC limma (raw): (430, 991) (3329 KB)

METADATA:

CD (n=215):
  Age  : 56.2 ± 8.2
  Sex  : male=111 female=104
  Batch: 0=189 1=26

UC (n=430):
  Age  : 58.7 ± 7.7
  Sex  : male=218 female=212
  Batch: 0=358 1=72

PROTEINS:
  Final protein count: 991
  Sample proteins: ['PLTP', 'PLXNA4', 'PLIN3'] ... ['CALCA', 'CALB2', 'CAPG']

IMPUTATION:
method  mean_ks  selected
   KNN 0.337982     False
Median 0.499760     False
  MICE 0.329980      True
  Selected: MICE

BATCH DERIVATION:
 CD_batch0  CD_batch1  UC_batch0  UC_batch1
       191         26        364         73

QUALITY CHECKS:
  CD NaN: False
  UC NaN: False
  CD mean: 0.043031 (expect ~0)
  CD std : 1.020478 (expect ~1)
  UC mean: -0.021516
  UC std : 0.984172

PIPELINE STEPS COMPLETED:
  ✅ Load data             : 658 × 2923
  ✅ Remove IBD-unclass    : 654 pat